In [1]:
import pandas as pd 
import numpy as np 
import json

# Bringing in server-side message timestamps from tajriba.json
For reasons such as latency and misconfigured client clocks, there can be a discrepancy between the client-side timestamp recording on the message object (`ts`), and the server-side timestamp of when the message was sent and processed. For analysis that rely on the order of messages, the server-based timestamp is more reliable. The code below illustrates how to extract the server-side timestamps from the `tajriba.json` file containing the experimental data. 

This snippet runs with sample data generated in a local instance by the authors (using `empirica export` and a copy of `.empirica/local/tajriba.json`), so the differences between `ts` and `server_ts` may be minimal. However, the differences can be quite large when conducting experiments with online groups. 

## Load game-level data and tajriba.json logs

In [2]:
# Game-level data 
df_games = pd.read_csv("./example_grail_data/game.csv")

In [3]:
# Tajriba logs, filtered for chat messages
tajriba_json = [json.loads(x) for x in open("./example_grail_data/tajriba.json", "r").readlines()][2:]
tajriba_chats = [x for x in tajriba_json if x["kind"] == "Attribute" and x["obj"]["key"] == "chat"]
df_tajriba_chat = pd.DataFrame([dict(createdAt=x["obj"]["createdAt"], **json.loads(x["obj"]["val"])) for x in tajriba_chats])
df_tajriba_chat["server_ts"] = pd.to_datetime(df_tajriba_chat["createdAt"]).astype('int64') // 10**6
df_tajriba_chat["text"] = df_tajriba_chat["text"].str.strip()
df_tajriba_chat["sender"] = df_tajriba_chat["sender"].astype(str)

# Join the server-side timestamps to the transcript for a single game, then write the transcript with server-side timestamps to json in a new column 
This example is applied to a single game for demonstration, but can be wrapped in a function and used to process a series of games using `pd.Series.apply`. 

In [4]:
# Create a dataframe from the chat object 
df_chat = pd.DataFrame(json.loads(df_games["chat"].values[0])).assign(sender = lambda x: x.sender.astype(str))

In [5]:
# Merge on text, client timestamp, and sender metadata
df_corrected_ts = df_chat.merge(df_tajriba_chat[["text", "ts","sender", "server_ts"]], on=["text", "ts", "sender"], how="left")

# Confirm that the resulting dataframe has expected length and no null timestamps 
assert(len(df_chat) == len(df_corrected_ts) and all(~df_corrected_ts["server_ts"].isnull()))

In [6]:
df_corrected_ts.head()

,text,ts,sender,server_ts
0,@[Facilitator] how should we get started?,1770651182844,"{'id': '01KH1GCNVN7JNMWNBYD8JPA4G7', 'name': '...",1770651182846
1,Let's start by having each person share any in...,1770651188115,"{'id': 'ai', 'name': 'Facilitator', 'avatar': ...",1770651188119
2,@[Facilitator] how much time do we have left?,1770651215198,"{'id': '01KH1GCNVN7JNMWNBYD8JPA4G7', 'name': '...",1770651215200
3,@[Green] We have 9 minutes left to make our de...,1770651218398,"{'id': 'ai', 'name': 'Facilitator', 'avatar': ...",1770651218403
4,"Ok, I suggest we share our information. Eldoro...",1770651241632,"{'id': '01KH1GCNVN7JNMWNBYD8JPA4G7', 'name': '...",1770651241633


In [7]:
# Create a new column with the server-timestamped transcript in json 
df_games["chat_corrected_ts"] = df_corrected_ts.to_json()